In [ ]:
# ============================================================
# Масштабное соответствие OSM-гидрографии
# ============================================================
#
# Геометрия: Precision / Recall / F1
# Структура: плотность, частота, KS, Pearson
# P25/P50/P75/P90 — описательная статистика длин
#
# Охват = bbox топокарты 1:200k
# Буфер = 100 м
# Ячейка = 500 м
# CRS задаётся для каждого участка

import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from shapely.geometry import box
from scipy.stats import ks_2samp, pearsonr
2
warnings.filterwarnings("ignore")


# ============================================================
# Настройки
# ============================================================

BUFFER = 100
CELL_SIZE = 500

SCALES = ['1:200k', '1:500k', '1:1M']


# ============================================================
# Статистика слоя
# ============================================================

def layer_stats(gdf, area_km2):
    """Рассчитывает основные характеристики слоя."""

    gdf = gdf[
        gdf.geometry.notna() &
        ~gdf.geometry.is_empty
    ].copy()

    lengths = gdf.geometry.length.to_numpy()
    lengths = lengths[
        np.isfinite(lengths) &
        (lengths > 0)
    ]

    if len(lengths) == 0:
        return {
            'objects': 0,
            'length_km': 0.0,
            'density': 0.0,
            'frequency': 0.0,
            'lengths': np.array([]),
            'p25': np.nan,
            'p50': np.nan,
            'p75': np.nan,
            'p90': np.nan
        }

    return {
        'objects': len(lengths),
        'length_km': lengths.sum() / 1000,
        'density': lengths.sum() / 1000 / area_km2,
        'frequency': len(lengths) / area_km2,
        'lengths': lengths,
        'p25': np.percentile(lengths, 25),
        'p50': np.percentile(lengths, 50),
        'p75': np.percentile(lengths, 75),
        'p90': np.percentile(lengths, 90)
    }


# ============================================================
# Геометрическое соответствие
# ============================================================

def geometry_metrics(topo, osm, buffer_m=BUFFER):
    """Рассчитывает Precision, Recall и F1."""

    topo_len = float(topo.length.sum())
    osm_len = float(osm.length.sum())

    if topo_len <= 0 or osm_len <= 0:
        return np.nan, np.nan, np.nan

    topo_buffer = topo.geometry.buffer(buffer_m).union_all()
    osm_buffer = osm.geometry.buffer(buffer_m).union_all()

    osm_match = osm.geometry.intersection(topo_buffer).length.sum()
    precision = osm_match / osm_len

    topo_match = topo.geometry.intersection(osm_buffer).length.sum()
    recall = topo_match / topo_len

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else np.nan
    )

    return float(precision), float(recall), float(f1)


# ============================================================
# Относительное расстояние
# ============================================================

def relative_distance(a, b):
    """Рассчитывает нормированное различие двух показателей."""

    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan

    m = max(abs(a), abs(b))

    if m == 0:
        return 0.0

    return abs(a - b) / m


# ============================================================
# Критерий Колмогорова–Смирнова
# ============================================================

def ks_distance(osm_lengths, topo_lengths):
    """Сравнивает распределения длин объектов."""

    if len(osm_lengths) == 0 or len(topo_lengths) == 0:
        return np.nan

    return float(
        ks_2samp(osm_lengths, topo_lengths).statistic
    )


# ============================================================
# Построение сетки
# ============================================================

def make_grid(bbox, crs, cell_size=CELL_SIZE):
    """Создаёт регулярную сетку анализа."""

    minx, miny, maxx, maxy = bbox.bounds

    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    cells = [
        box(
            x, y,
            min(x + cell_size, maxx),
            min(y + cell_size, maxy)
        )
        for x in xs
        for y in ys
    ]

    return gpd.GeoDataFrame(
        {'cell_id': np.arange(len(cells))},
        geometry=cells,
        crs=crs
    )


# ============================================================
# Длина линий по ячейкам
# ============================================================

def lengths_by_grid(gdf, grid):
    """Рассчитывает длину линий внутри каждой ячейки."""

    lines = (
        gdf[['geometry']]
        .copy()
        .reset_index(drop=True)
    )

    lines['line_id'] = np.arange(len(lines))

    join = gpd.sjoin(
        grid,
        lines[['line_id', 'geometry']],
        how='inner',
        predicate='intersects'
    )

    values = np.zeros(len(grid), dtype=float)

    if join.empty:
        return values

    for cell_id, group in join.groupby('cell_id'):
        cell_id = int(cell_id)
        cell = grid.geometry.iloc[cell_id]
        total = 0.0

        for line_id in (
            group['line_id']
            .dropna()
            .astype(int)
            .unique()
        ):
            geom = lines.geometry.iloc[line_id]

            if geom is None or geom.is_empty:
                continue

            total += geom.intersection(cell).length

        values[cell_id] = total

    return values


# ============================================================
# Пространственная корреляция
# ============================================================

def spatial_pearson(topo, osm, bbox, cell_size=CELL_SIZE):
    """Рассчитывает корреляцию распределения длин по сетке."""

    grid = make_grid(bbox, topo.crs, cell_size)

    if len(grid) < 2:
        return np.nan

    topo_values = lengths_by_grid(topo, grid)
    osm_values = lengths_by_grid(osm, grid)

    if len(topo_values) < 2:
        return np.nan

    if np.std(topo_values) == 0 or np.std(osm_values) == 0:
        return np.nan

    try:
        return float(
            pearsonr(topo_values, osm_values).statistic
        )
    except Exception:
        return np.nan


# ============================================================
# Эмпирическая функция распределения
# ============================================================

def ecdf(values):
    """Возвращает координаты эмпирической функции распределения."""

    values = np.sort(np.asarray(values))

    if len(values) == 0:
        return np.array([]), np.array([])

    y = (np.arange(len(values)) + 1) / len(values)

    return values, y


# ============================================================
# Анализ одного участка
# ============================================================

def analyse_site(site):
    """Выполняет полный анализ одного участка."""

    name = site['name']
    crs = site['crs']

    print(f'\n{name}')
    print('=' * 60)

    # Охват
    ref = gpd.read_file(
        site['topo']['1:200k']
    ).to_crs(crs)

    if ref.empty:
        raise ValueError('Пустая карта 1:200k')

    bbox = box(*ref.total_bounds)
    area_km2 = bbox.area / 1e6

    print(
        f'Область: '
        f'{(bbox.bounds[2]-bbox.bounds[0])/1000:.1f} × '
        f'{(bbox.bounds[3]-bbox.bounds[1])/1000:.1f} км'
    )
    print(f'Площадь: {area_km2:.1f} км²')

    # OSM
    osm = gpd.read_file(site['osm']).to_crs(crs)
    osm = gpd.clip(osm, bbox)

    if osm.empty:
        raise ValueError('OSM не пересекает область')

    osm_s = layer_stats(osm, area_km2)

    print(
        f'OSM: {osm_s["objects"]} объектов, '
        f'{osm_s["length_km"]:.1f} км'
    )

    rows = []

    # Анализ масштабов
    for scale in SCALES:

        try:
            topo = (
                gpd.read_file(site['topo'][scale])
                .to_crs(crs)
            )
        except Exception as e:
            print(f'{scale}: ошибка чтения — {e}')
            continue

        topo = gpd.clip(topo, bbox)

        if topo.empty:
            print(f'{scale}: нет данных')
            continue

        ts = layer_stats(topo, area_km2)

        P, R, F1 = geometry_metrics(
            topo, osm, BUFFER
        )

        density_dist = relative_distance(
            osm_s['density'],
            ts['density']
        )

        frequency_dist = relative_distance(
            osm_s['frequency'],
            ts['frequency']
        )

        KS = ks_distance(
            osm_s['lengths'],
            ts['lengths']
        )

        Pearson = spatial_pearson(
            topo,
            osm,
            bbox,
            CELL_SIZE
        )

        rows.append({
            'site': name,
            'scale': scale,
            'precision': P,
            'recall': R,
            'f1': F1,
            'density_distance': density_dist,
            'frequency_distance': frequency_dist,
            'ks_distance': KS,
            'pearson': Pearson,
            'topo_objects': ts['objects'],
            'topo_length_km': ts['length_km'],
            'topo_density': ts['density'],
            'topo_frequency': ts['frequency'],
            'p25': ts['p25'],
            'p50': ts['p50'],
            'p75': ts['p75'],
            'p90': ts['p90']
        })

        print(f'\n{scale}')
        print(f'  Precision: {P:.3f}')
        print(f'  Recall:    {R:.3f}')
        print(f'  F1:        {F1:.3f}')
        print(f'  Objects:   {ts["objects"]}')
        print(f'  Density:   {ts["density"]:.4f}')
        print(f'  P50:       {ts["p50"]:.1f} м')
        print(f'  KS:        {KS:.3f}')
        print(f'  Pearson:   {Pearson:.3f}')

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError('Не удалось получить результаты')

    # Определение ближайшего масштаба
    def best_scale(column, mode='min'):
        """Возвращает масштаб с минимальным или максимальным значением."""

        x = df[['scale', column]].dropna()

        if x.empty:
            return None

        idx = (
            x[column].idxmin()
            if mode == 'min'
            else x[column].idxmax()
        )

        return x.loc[idx, 'scale']

    summary = {
        'site': name,
        'crs': crs,
        'area_km2': area_km2,
        'osm_objects': osm_s['objects'],
        'osm_length_km': osm_s['length_km'],
        'osm_density': osm_s['density'],
        'geometry_scale': best_scale('f1', 'max'),
        'density_scale': best_scale('density_distance', 'min'),
        'frequency_scale': best_scale('frequency_distance', 'min'),
        'length_scale': best_scale('ks_distance', 'min'),
        'spatial_scale': best_scale('pearson', 'max')
    }

    return df, summary, osm_s, bbox


# ============================================================
# Графики
# ============================================================

def make_plots(profile, output_dir='output'):
    """Строит сводные тепловые карты показателей."""

    os.makedirs(output_dir, exist_ok=True)

    metrics = {
        'f1': 'F1 — геометрическая близость',
        'density_distance': 'Расстояние — плотность',
        'frequency_distance': 'Расстояние — частота',
        'ks_distance': 'KS — распределение длин',
        'pearson': 'Pearson — пространственная структура'
    }

    for metric, title in metrics.items():

        table = profile.pivot(
            index='site',
            columns='scale',
            values=metric
        )

        plt.figure(
            figsize=(8, max(6, len(table) * 0.30))
        )

        sns.heatmap(
            table,
            annot=True,
            fmt='.2f',
            cmap='viridis',
            linewidths=.3,
            linecolor='white',
            cbar_kws={'label': metric}
        )

        plt.title(title)
        plt.xlabel('Масштаб')
        plt.ylabel('Участок')
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                output_dir,
                f'heatmap_{metric}.png'
            ),
            dpi=180,
            bbox_inches='tight'
        )

        plt.close()


# ============================================================
# ECDF
# ============================================================

def plot_ecdf(
    osm_lengths,
    topo_lengths,
    scale,
    site_name,
    output_dir='output'
):
    """Строит ECDF распределения длин для участка."""

    x1, y1 = ecdf(osm_lengths)
    x2, y2 = ecdf(topo_lengths)

    plt.figure(figsize=(8, 6))

    if len(x1):
        plt.step(
            x1, y1,
            where='post',
            label='OSM'
        )

    if len(x2):
        plt.step(
            x2, y2,
            where='post',
            label=scale
        )

    plt.xlabel('Длина линейного объекта, м')
    plt.ylabel('Эмпирическая функция распределения')
    plt.title(f'{site_name}: OSM vs {scale}')
    plt.grid(alpha=.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            output_dir,
            f'ecdf_{site_name}_{scale}.png'
        ),
        dpi=180,
        bbox_inches='tight'
    )

    plt.close()


# ============================================================
# Анализ всех участков
# ============================================================

def run_all(sites, output_dir='output'):
    """Запускает анализ всех участков и сохраняет результаты."""

    os.makedirs(output_dir, exist_ok=True)

    all_details = []
    summaries = []

    for site in sites:

        try:
            df, summary, osm_s, bbox = analyse_site(site)

            all_details.append(df)
            summaries.append(summary)

        except Exception as e:
            print(f'\nERROR {site["name"]}: {e}')

    if not all_details:
        print('Нет успешно обработанных участков.')
        return None, None

    details = pd.concat(
        all_details,
        ignore_index=True
    )

    summary = pd.DataFrame(summaries)

    # Сводный профиль
    scale_profile = summary[[
        'site',
        'geometry_scale',
        'density_scale',
        'frequency_scale',
        'length_scale',
        'spatial_scale'
    ]].copy()

    # Геометрия
    geometry = details[[
        'site',
        'scale',
        'precision',
        'recall',
        'f1'
    ]].copy()

    # Структура
    structure = details[[
        'site',
        'scale',
        'density_distance',
        'frequency_distance',
        'ks_distance',
        'pearson'
    ]].copy()

    # Описательная статистика
    descriptive = details[[
        'site',
        'scale',
        'topo_objects',
        'topo_length_km',
        'topo_density',
        'topo_frequency',
        'p25',
        'p50',
        'p75',
        'p90'
    ]].copy()

    # Сохранение результатов
    tables = {
        'scale_profile.csv': scale_profile,
        'geometry_metrics.csv': geometry,
        'structure_metrics.csv': structure,
        'descriptive_statistics.csv': descriptive,
        'full_summary.csv': summary
    }

    for filename, table in tables.items():
        table.to_csv(
            os.path.join(output_dir, filename),
            index=False,
            encoding='utf-8-sig'
        )

    # Графики
    make_plots(details, output_dir)

    # Краткий вывод
    print('\n' + '=' * 80)
    print('Профиль масштабного соответствия')
    print('=' * 80)

    print('\nГеометрическая близость')
    print(geometry.to_string(index=False))

    print('\nСтруктурная близость')
    print(structure.to_string(index=False))

    print('\nИтоговый профиль')
    print(scale_profile.to_string(index=False))

    print('\nФайлы сохранены в:', output_dir)

    return scale_profile, details


# ============================================================
# Запуск
# ============================================================

if __name__ == '__main__':

    sites = [

        {
            'name': 'Алдан',
            'crs': 'EPSG:28421',
            'osm':
                '/content/drive/MyDrive/Experiments/Aldan/aldan_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Aldan/aldan_topo_200k.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Aldan/aldan_topo_500k_2.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Aldan/aldan_topo_1000k.shp'
            }
        },
        {
            'name': 'Березники',
            'crs': 'EPSG:28410',
            'osm':
                '/content/drive/MyDrive/Experiments/Berezniki/berezniki_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Berezniki/berezniki_topo_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Berezniki/berezniki_topo_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Berezniki/berezniki_topo_1000.shp'
            }
        },
        {
            'name': 'Бодайбо',
            'crs': 'EPSG:28419',
            'osm':
                '/content/drive/MyDrive/Experiments/Bodaibo/bodaibo_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Bodaibo/bodaibo_topo_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Bodaibo/bodaibo_topo_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Bodaibo/bodaibo_topo_1000.shp'
            }
        },
        {
            'name': 'Братск',
            'crs': 'EPSG:28417',
            'osm':
                '/content/drive/MyDrive/Experiments/Bratsk/bratsk_topo.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Bratsk/bratsk_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Bratsk/bratsk_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Bratsk/bratsk_1000.shp'
            }
        },
        {
            'name': 'Чита',
            'crs': 'EPSG:28419',
            'osm':
                '/content/drive/MyDrive/Experiments/Chita/chita_topo.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Chita/chita_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Chita/chita_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Chita/chita_1000.shp'
            }
        },
        {
            'name': 'Элиста',
            'crs': 'EPSG:28408',
            'osm':
                '/content/drive/MyDrive/Experiments/Elista/elista_topo.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Elista/elista_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Elista/elista_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Elista/elista_1000.shp'
            }
        },
        {
            'name': 'Грозный',
            'crs': 'EPSG:28408',
            'osm':
                '/content/drive/MyDrive/Experiments/Grozniy/grozniy_topo.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Grozniy/grozniy_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Grozniy/grozniy_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Grozniy/grozniy_1000.shp'
            }
        },
        {
            'name': 'Иркутск',
            'crs': 'EPSG:28418',
            'osm':
                '/content/drive/MyDrive/Experiments/Irkutsk/irkutsk_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Irkutsk/irkutsk_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Irkutsk/irkutsk_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Irkutsk/irkutsk_1000.shp'
            }
        },
        {
            'name': 'Кисловодск',
            'crs': 'EPSG:28408',
            'osm':
                '/content/drive/MyDrive/Experiments/Kislovodsk/kislo_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Kislovodsk/kislo_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Kislovodsk/kislo_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Kislovodsk/kislo_1000.shp'
            }
        },
        {
            'name': 'Комсомольск-на-Амуре',
            'crs': 'EPSG:28423',
            'osm':
                '/content/drive/MyDrive/Experiments/Kmsk_na_amure/kmsk_na_amure_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Kmsk_na_amure/kmsk_na_amure_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Kmsk_na_amure/kmsk_na_amure_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Kmsk_na_amure/kmsk_na_amure_1000.shp'
            }
        },
        {
            'name': 'Краснодар',
            'crs': 'EPSG:28407',
            'osm':
                '/content/drive/MyDrive/Experiments/Krasnodar/krasn_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Krasnodar/krasn_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Krasnodar/krasn_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Krasnodar/krasn_1000.shp'
            }
        },
        {
            'name': 'Курган',
            'crs': 'EPSG:28411',
            'osm':
                '/content/drive/MyDrive/Experiments/Kurgan/kurg_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Kurgan/kurg_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Kurgan/kurg_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Kurgan/kurg_1000.shp'
            }
        },
        {
            'name': 'Кызыл',
            'crs': 'EPSG:28416',
            'osm':
                '/content/drive/MyDrive/Experiments/Kyzyl/kyzyl_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Kyzyl/kyzyl_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Kyzyl/kyzyl_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Kyzyl/kyzyl_1000.shp'
            }
        },
        {
            'name': 'Лесосибирск',
            'crs': 'EPSG:28416',
            'osm':
                '/content/drive/MyDrive/Experiments/Lesosibirsk/les_sibiri_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Lesosibirsk/les_sibiri_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Lesosibirsk/les_sibiri_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Lesosibirsk/les_sibiri_1000.shp'
            }
        },
        {
            'name': 'Мариинск',
            'crs': 'EPSG:28415',
            'osm':
                '/content/drive/MyDrive/Experiments/Mariinsk/mariinsk_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Mariinsk/mariinsk_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Mariinsk/mariinsk_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Mariinsk/mariinsk_1000.shp'
            }
        },
        {
            'name': 'Москва',
            'crs': 'EPSG:28407',
            'osm':
                '/content/drive/MyDrive/Experiments/Moscow/msk_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Moscow/msk_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Moscow/msk_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Moscow/msk_1000.shx'
            }
        },
         {
            'name': 'Мурманск',
            'crs': 'EPSG:28406',
            'osm':
                '/content/drive/MyDrive/Experiments/Murmansk/murmansk_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Murmansk/murmansk_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Murmansk/murmansk_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Murmansk/murmansk_1000.shp'
            }
        },
        {
            'name': 'Нарьян-Мар',
            'crs': 'EPSG:28409',
            'osm':
                '/content/drive/MyDrive/Experiments/Narian_Mar/narian_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Narian_Mar/narian_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Narian_Mar/narian_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Narian_Mar/narian_1000.shp'
            }
        },
        {
            'name': 'Нижневартовск',
            'crs': 'EPSG:28413',
            'osm':
                '/content/drive/MyDrive/Experiments/Niznevartovsk/niznevar_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Niznevartovsk/niznevar_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Niznevartovsk/niznevar_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Niznevartovsk/niznevar_1000.shp'
            }
        },
        {
            'name': 'Петропавловск-Камчатский',
            'crs': 'EPSG:28427',
            'osm':
                '/content/drive/MyDrive/Experiments/Petropavlosk_Kamchatskiy/pet_kamchat_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Petropavlosk_Kamchatskiy/pet_kamchat_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Petropavlosk_Kamchatskiy/pet_kamchat_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Petropavlosk_Kamchatskiy/pet_kamchat_1000.shp'
            }
        },
        {
            'name': 'Петрозаводск',
            'crs': 'EPSG:28406',
            'osm':
                '/content/drive/MyDrive/Experiments/Petrozavodsk/petro_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Petrozavodsk/petro_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Petrozavodsk/petro_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Petrozavodsk/petro_1000.shp'
            }
        },
        {
            'name': 'Рубцовск',
            'crs': 'EPSG:28414',
            'osm':
                '/content/drive/MyDrive/Experiments/Rubtsovsk/rubts_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Rubtsovsk/rubts_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Rubtsovsk/rubts_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Rubtsovsk/rubts_1000.shp'
            }
        },
        {
            'name': 'Саратов',
            'crs': 'EPSG:28408',
            'osm':
                '/content/drive/MyDrive/Experiments/Saratov/saratov_osm_2.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Saratov/saratov_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Saratov/saratov_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Saratov/saratov_1000.shp'
            }
        },
        {
            'name': 'Санкт-Петербург',
            'crs': 'EPSG:28406',
            'osm':
                '/content/drive/MyDrive/Experiments/Saint_Petersburg/spb_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Saint_Petersburg/spb_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Saint_Petersburg/spb_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Saint_Petersburg/spb_1000.shp'
            }
        },
        {
            'name': 'Уфа',
            'crs': 'EPSG:28410',
            'osm':
                '/content/drive/MyDrive/Experiments/Ufa/Ufa_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Ufa/Ufa_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Ufa/Ufa_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Ufa/Ufa_1000.shp'
            }
        },
        {
            'name': 'Великий Устюг',
            'crs': 'EPSG:28408',
            'osm':
                '/content/drive/MyDrive/Experiments/Velikiy_Ustug/velikiy_ustug_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Velikiy_Ustug/velikiy_ustug_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Velikiy_Ustug/velikiy_ustug_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Velikiy_Ustug/velikiy_ustug_1000.shp'
            }
        },
        {
            'name': 'Владивосток',
            'crs': 'EPSG:28422',
            'osm':
                '/content/drive/MyDrive/Experiments/Vladivostok/vladik_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Vladivostok/vladik_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Vladivostok/vladik_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Vladivostok/vladik_1000.shp'
            }
        },
        {
            'name': 'Воркута',
            'crs': 'EPSG:28411',
            'osm':
                '/content/drive/MyDrive/Experiments/Vorkuta/Vorkuta_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Vorkuta/Vorkuta_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Vorkuta/Vorkuta_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Vorkuta/Vorkuta_1000.shp'
            }
        },
        {
            'name': 'Воронеж',
            'crs': 'EPSG:28407',
            'osm':
                '/content/drive/MyDrive/Experiments/Voronezh/voronezh_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Voronezh/voronezh_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Voronezh/voronezh_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Voronezh/voronezh_1000.shp'
            }
        },
        {
            'name': 'Южно-Сахалинск',
            'crs': 'EPSG:28424',
            'osm':
                '/content/drive/MyDrive/Experiments/Yuzno-Sahalinsk/yuzno_osm.shp',
            'topo': {
                '1:200k':
                    '/content/drive/MyDrive/Experiments/Yuzno-Sahalinsk/yuzno_200.shp',
                '1:500k':
                    '/content/drive/MyDrive/Experiments/Yuzno-Sahalinsk/yuzno_500.shp',
                '1:1M':
                    '/content/drive/MyDrive/Experiments/Yuzno-Sahalinsk/yuzno_1000.shp'
            }
        }

    ]

    scale_profile, details = run_all(
        sites,
        output_dir='output'
    )

    print('\n' + '=' * 80)
    print('Готово')
    print('=' * 80)


Алдан
Область: 103.2 × 103.4 км
Площадь: 10671.4 км²
OSM: 377 объектов, 2274.7 км

1:200k
  Precision: 0.544
  Recall:    0.293
  F1:        0.381
  Objects:   2086
  Density:   0.3919
  P50:       1492.5 м
  KS:        0.324
  Pearson:   0.500

1:500k
  Precision: 0.281
  Recall:    0.199
  F1:        0.233
  Objects:   701
  Density:   0.2950
  P50:       2489.7 м
  KS:        0.177
  Pearson:   0.408

1:1M
  Precision: 0.221
  Recall:    0.256
  F1:        0.237
  Objects:   314
  Density:   0.1763
  P50:       4342.7 м
  KS:        0.186
  Pearson:   0.417

Березники
Область: 101.1 × 100.5 км
Площадь: 10162.4 км²
OSM: 1533 объектов, 4089.0 км

1:200k
  Precision: 0.498
  Recall:    0.384
  F1:        0.434
  Objects:   2334
  Density:   0.4851
  P50:       1676.6 м
  KS:        0.416
  Pearson:   0.542

1:500k
  Precision: 0.320
  Recall:    0.451
  F1:        0.374
  Objects:   290
  Density:   0.2533
  P50:       5811.7 м
  KS:        0.597
  Pearson:   0.566

1:1M
  Precision: 